# Monitoring
## Layer 4 — 배포 이후 운영 안정성을 3층 구조로 추적

## 목적

이 노트북은 운영 중인 정책 또는 모델의 안정성을 점검하고, 데이터 변화와 성능 저하가 의사결정에 미치는 영향을 조기에 파악하기 위한 모니터링 기준을 정리합니다. `01_data_mart.ipynb`에서 이미 확인했듯, 이 데이터셋에는 절대 시점 정보가 없어 전통적인 시계열 분석은 적용할 수 없기에, `SK_ID_CURR`를 기준으로 가상의 운영 순서를 구성하고 이를 배포 후 시나리오(Pseudo Production)로 해석했습니다. 이를 바탕으로 입력 데이터 변화, 정책 또는 모델 출력의 안정성, 그리고 최종 의사결정 결과를 단계적으로 점검하는 3층 모니터링 구조를 설계합니다.

## Step 1. Pseudo Production 구성

절대 시점 정보가 없어 전통적 시계열 기반 검증이 어려웠고, 이를 보완하기 위해 배포 이후를 가정한 Pseudo Production 환경을 구성합니다. 이 환경에서 정책 변화에 따른 입력 분포 변화, 모델·정책 출력의 안정성, 의사결정 결과의 민감도를 점검합니다. 판단 기준은 baseline 대비 변화폭, 시나리오별 견고성, 그리고 운영 관점에서 허용 가능한 변동 범위입니다. 이를 통해 각 정책이 배포 후 환경에서도 안정적으로 해석 가능한지 검증합니다.

`val_raw`는 모델 개발, calibration, 정책 설계에 사용한 데이터지만 절대 시점 정보가 없어 실제 배포 이후 유입 순서를 직접 재현할 수는 없었습니다. 이에 따라 `SK_ID_CURR`를 기준으로 5개의 연속 구간을 나누고, 이를 배포 이후 순차적으로 유입된 신청자 집단을 가정한 pseudo production 시나리오로 해석했습니다. 이 가정 아래에서 정책 변화에 따른 분포와 출력 안정성을 점검했습니다.

PSI 등 분포 비교의 기준은 `val_raw`가 아니라 `train_raw`로 두었습니다. PSI는 운영 중 입력이 모델이 학습한 분포에서 얼마나 벗어났는지를 보는 지표이므로, 모델이 실제로 학습한 `train_raw`를 baseline으로 삼는 것이 삼았습니다.

In [2]:
import numpy as np
import pandas as pd

train_raw = pd.read_csv('../data/train_raw.csv')
val_raw   = pd.read_csv('../data/val_raw.csv')
policy_df = pd.read_csv('../data/policy_simulation.csv')  # 05_policy_simulation.ipynb 산출물

id_col = 'SK_ID_CURR' if 'SK_ID_CURR' in val_raw.columns else val_raw.columns[0]

monitor_df = val_raw.merge(
    policy_df[[id_col, 'PD_calibrated', 'risk_tier', 'approved']],
    on=id_col, how='inner'
)

# SK_ID_CURR 순서로 5개 배치 분할
monitor_df = monitor_df.sort_values(id_col).reset_index(drop=True)
N_BATCHES = 5
monitor_df['batch'] = pd.qcut(monitor_df.index, N_BATCHES, labels=[f'Batch {i+1}' for i in range(N_BATCHES)])

batch_sizes = monitor_df.groupby('batch', observed=True).size()
print("Pseudo Production 배치 구성:")
print(batch_sizes)
print(f"\n총 {len(monitor_df):,}명, 베이스라인(train_raw) {len(train_raw):,}명")


Pseudo Production 배치 구성:
batch
Batch 1    12301
Batch 2    12300
Batch 3    12301
Batch 4    12300
Batch 5    12301
dtype: int64

총 61,503명, 베이스라인(train_raw) 246,008명


## Step 2. Layer 1 — 입력 분포 변화 (PSI)

PSI(Population Stability Index)는 기준 분포와 현재 분포의 차이를 수치화한 지표이며, 값이 클수록 입력 분포 변화가 크다고 해석합니다. 일반적으로 `0.1 미만`은 안정, `0.1 이상`은 변화 징후, `0.25 이상`은 큰 변화로 해석되며 이 경우에는 원인 파악이 필요합니다. 정책 판단에 직접 관여하는 핵심 변수(EXT_SOURCE_2, DTI, CIR, LTV, AMT_INCOME_TOTAL, PD_calibrated)를 대상으로, 각 batch의 분포가 `train_raw` 대비 얼마나 변했는지 측정합니다.

In [3]:
def compute_psi(baseline, current, n_bins=10):
    bin_edges = np.quantile(baseline.dropna(), np.linspace(0, 1, n_bins + 1))
    bin_edges = np.unique(bin_edges)
    if len(bin_edges) < 3:
        return np.nan
    base_counts, _ = np.histogram(baseline.dropna(), bins=bin_edges)
    curr_counts, _ = np.histogram(current.dropna(), bins=bin_edges)
    base_pct = np.clip(base_counts / base_counts.sum(), 1e-4, None)
    curr_pct = np.clip(curr_counts / curr_counts.sum(), 1e-4, None)
    return float(np.sum((curr_pct - base_pct) * np.log(curr_pct / base_pct)))

PSI_FEATURES = ['EXT_SOURCE_2', 'DTI', 'CIR', 'LTV', 'AMT_INCOME_TOTAL', 'PD_calibrated']

psi_rows = []
for batch_name in batch_sizes.index:
    batch_df = monitor_df[monitor_df['batch'] == batch_name]
    row = {'batch': batch_name}
    for feat in PSI_FEATURES:
        base_series = train_raw[feat] if feat in train_raw.columns else monitor_df[feat]
        row[feat] = compute_psi(base_series, batch_df[feat])
    psi_rows.append(row)

psi_summary = pd.DataFrame(psi_rows).set_index('batch').round(4)
print("Batch별 PSI (train_raw 베이스라인 대비):")
print(psi_summary.to_string())

psi_flag = (psi_summary >= 0.25).any(axis=1)
print(f"\nPSI 0.25 이상 발생 batch: {psi_flag[psi_flag].index.tolist() if psi_flag.any() else '없음'}")


Batch별 PSI (train_raw 베이스라인 대비):
         EXT_SOURCE_2     DTI     CIR     LTV  AMT_INCOME_TOTAL  PD_calibrated
batch                                                                         
Batch 1        0.0005  0.0008  0.0006  0.0009            0.0002         0.0008
Batch 2        0.0005  0.0010  0.0003  0.0008            0.0006         0.0011
Batch 3        0.0006  0.0008  0.0011  0.0010            0.0012         0.0003
Batch 4        0.0009  0.0008  0.0005  0.0005            0.0008         0.0004
Batch 5        0.0007  0.0010  0.0015  0.0015            0.0002         0.0006

PSI 0.25 이상 발생 batch: 없음


Batch 1~5 전 구간의 PSI는 0.0002~0.0015로 경고 기준과 위험 기준에 한참 못 미쳐, 입력 분포의 유의미한 이탈은 관찰되지 않았습니다. 또한 배치가 진행될수록 단조 증가하는 구조적 드리프트도 보이지 않고, 피처별 변동은 노이즈 수준에 머물렀습니다.

피처별로 보면 `CIR`과 `LTV`가 Batch 5에서 각각 0.0015로 가장 크게 변동했고, `AMT_INCOME_TOTAL`은 Batch 3에서 0.0012의 국소적 피크를 보였습니다. `PD_calibrated`는 Batch 2에서 0.0011로 가장 컸지만, 원본 입력 변수들의 변동폭 범위 안에 있어 불안정 신호로 보기는 어렵습니다. 원본 변수와 모델 출력 모두에서 이상 이동을 시사하는 패턴은 관찰되지 않았습니다.

따라서 관찰 기간 내 입력 분포와 모델 출력은 운영 관점에서 안정적으로 유지된 것으로 판단합니다.

## Step 3. Layer 2 — 점수·정책 라벨 분포 변화

PSI가 안정적이더라도, PD 점수 자체나 그로부터 나온 정책 판단(tier)이 쏠릴 수 있습니다. 이 층에서는 각 batch의 `risk_tier` 비중과 평균 PD가 각 batch의 risk_tier 비중과 평균 PD가 정책 설계 기준 분포 대비 얼마나 벌어졌는지 확인합니다. 이는 PSI처럼 입력 변화 자체를 보는 지표가 아니라, 그 결과 승인·거절 의사결정이 특정 방향으로 쏠리는지를 확인하는 지표입니다.

In [4]:
baseline_tier_dist = policy_df['risk_tier'].value_counts(normalize=True)
baseline_approval_rate = policy_df['approved'].mean()
baseline_avg_pd = policy_df['PD_calibrated'].mean()

tier_drift_rows = []
for batch_name in batch_sizes.index:
    batch_df = monitor_df[monitor_df['batch'] == batch_name]
    batch_tier_dist = batch_df['risk_tier'].value_counts(normalize=True).reindex(baseline_tier_dist.index, fill_value=0)
    max_tier_shift = (batch_tier_dist - baseline_tier_dist).abs().max()
    tier_drift_rows.append({
        'batch': batch_name,
        'avg_PD': batch_df['PD_calibrated'].mean(),
        '승인율': batch_df['approved'].mean(),
        '최대_tier_비중_변화(%p)': max_tier_shift * 100,
    })

tier_drift_summary = pd.DataFrame(tier_drift_rows).set_index('batch').round(4)
print(f"기준(val_raw 전체): 평균 PD {baseline_avg_pd:.4f}, 승인율 {baseline_approval_rate:.4f}")
print()
print(tier_drift_summary.to_string())

TIER_SHIFT_THRESHOLD = 0.05  # tier 비중이 5%p 이상 벌어지면 주의
tier_flag = tier_drift_summary['최대_tier_비중_변화(%p)'] >= TIER_SHIFT_THRESHOLD * 100
print(f"\ntier 비중 5%p 이상 이동한 batch: {tier_flag[tier_flag].index.tolist() if tier_flag.any() else '없음'}")


기준(val_raw 전체): 평균 PD 0.0806, 승인율 0.7089

         avg_PD     승인율  최대_tier_비중_변화(%p)
batch                                     
Batch 1  0.0802  0.7146             0.5242
Batch 2  0.0818  0.7006             0.4812
Batch 3  0.0809  0.7053             0.5702
Batch 4  0.0805  0.7124             0.3675
Batch 5  0.0797  0.7114             0.4539

tier 비중 5%p 이상 이동한 batch: 없음


평균 PD와 승인율은 기준값 주변의 좁은 범위(각각 0.0806, 0.7089)에서만 움직였고, Batch 1~5 구간에서 단조 증가 또는 감소 추세는 관찰되지 않았습니다. 즉, 관찰된 변동은 운영상 의미 있는 드리프트라기보다 잡음 수준의 변동으로 해석됩니다.

이는 PSI 결과와도 일관됩니다. `EXT_SOURCE_2`와 같은 피처를 포함한 주요 변수들에서 특정 batch의 뚜렷한 이탈이 없었고, `PD_calibrated` 역시 원본 변수들과 비슷한 범위의 PSI를 보여 신청자 신용 이력 구성 자체에 구조적 변화가 있었다고 보기는 어렵습니다. 이는 평균 PD·승인율의 배치 간 편차가 표본 구성에 따른 자연스러운 변동임을 시사합니다.

`최대_tier_비중_변화`도 Batch 3에서 0.5702%p로 가장 컸지만, 전 구간에서 5%p 기준을 크게 밑돌아 tier 간 쏠림은 관찰되지 않았습니다. 따라서 위험 구간별 승인 분포는 안정적으로 유지되었다고 판단합니다.

## Step 4. Layer 3 — 지연 라벨 기반 성능 추적

실제 운영에서는 대출 실행 후 부도 여부가 몇 달에서 몇 년 뒤에야 확인되기 때문에, 배포 시점에 즉시 성능을 관찰할 수 없습니다. 이 시차가 지연 라벨이며, 사후에 확인 가능한 TARGET을 이용해 뒤늦게 들어온 라벨이 당시 관측되었을 때의 성능을 재현할 수 있습니다.

추적 지표는 확정한 기준선과 동일한 AUC, KS, ECE이며, 이 중 하나라도 기준선 대비 뚜렷하게 저하되면 PSI와 tier 분포가 안정적이더라도 모델 성능 저하로 판단합니다.

In [5]:
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp

def compute_ece(y_true, y_prob, n_bins=10):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        mask = (y_prob >= lo) & (y_prob < hi) if i < n_bins - 1 else (y_prob >= lo) & (y_prob <= hi)
        n = mask.sum()
        if n == 0:
            continue
        ece += (n / len(y_prob)) * abs(y_true[mask].mean() - y_prob[mask].mean())
    return ece

BASELINE_AUC, BASELINE_KS, BASELINE_ECE = 0.7689, 0.3969, 0.0061  # 04_calibration.ipynb 확정값

perf_rows = []
for batch_name in batch_sizes.index:
    batch_df = monitor_df[monitor_df['batch'] == batch_name]
    y_true = batch_df['TARGET'].values
    y_prob = batch_df['PD_calibrated'].values
    auc = roc_auc_score(y_true, y_prob)
    ks, _ = ks_2samp(y_prob[y_true == 1], y_prob[y_true == 0])
    ece = compute_ece(y_true, y_prob)
    perf_rows.append({
        'batch': batch_name, 'n': len(batch_df), '부도율': y_true.mean(),
        'AUC': auc, 'KS': ks, 'ECE': ece,
        'AUC_변화': auc - BASELINE_AUC, 'ECE_변화': ece - BASELINE_ECE,
    })

perf_summary = pd.DataFrame(perf_rows).set_index('batch').round(4)
print(f"기준선(04_calibration.ipynb): AUC {BASELINE_AUC}, KS {BASELINE_KS}, ECE {BASELINE_ECE}")
print()
print(perf_summary.to_string())

AUC_DROP_THRESHOLD = 0.02   # AUC가 기준선 대비 0.02 이상 하락
ECE_RISE_THRESHOLD = 0.02   # ECE가 기준선 대비 0.02 이상 상승
perf_flag = (perf_summary['AUC_변화'] <= -AUC_DROP_THRESHOLD) | (perf_summary['ECE_변화'] >= ECE_RISE_THRESHOLD)
print(f"\n성능 저하 기준 초과 batch: {perf_flag[perf_flag].index.tolist() if perf_flag.any() else '없음'}")


기준선(04_calibration.ipynb): AUC 0.7689, KS 0.3969, ECE 0.0061

             n     부도율     AUC      KS     ECE  AUC_변화  ECE_변화
batch                                                         
Batch 1  12301  0.0800  0.7726  0.4148  0.0051  0.0037 -0.0010
Batch 2  12300  0.0810  0.7691  0.3974  0.0096  0.0002  0.0035
Batch 3  12301  0.0839  0.7625  0.3923  0.0050 -0.0064 -0.0011
Batch 4  12300  0.0760  0.7649  0.3991  0.0079 -0.0040  0.0018
Batch 5  12301  0.0828  0.7755  0.4133  0.0055  0.0066 -0.0006

성능 저하 기준 초과 batch: 없음


5개 batch 전 구간에서 입력 분포·정책 판단·모델 성능이 함께 안정적으로 유지되며, 뚜렷한 주의 신호는 관찰되지 않습니다. 따라서 관찰 기간 동안 구조적 드리프트는 확인되지 않았습니다.

`EXT_SOURCE_2`, `DTI`, `CIR`, `LTV`, `AMT_INCOME_TOTAL`, `PD_calibrated` 전 피처의 PSI는 0.0002~0.0015 수준으로, 경고 기준 0.1과 위험 기준 0.25를 크게 하회했습니다. 또한 Batch가 진행될수록 특정 방향으로 커지는 추세도 관찰되지 않아, 신청자 구성과 모델 출력 분포 모두 배포 시점 대비 안정적으로 유지된 것으로 판단합니다.

평균 PD(0.0797~0.0818)와 승인율(0.7006~0.7146)은 기준값(0.0806, 0.7089) 근처에서 소폭 등락할 뿐 단조 추세가 없고, `최대_tier_비중_변화`도 최대 0.5702%p(Batch 3)로 5%p 기준을 크게 밑돕니다. 즉 정책 threshold가 특정 위험 구간으로 쏠리는 현상은 관찰되지 않습니다.

AUC와 KS는 기준선(0.7689/0.3969) 근처에서 유지되어 변별력 저하는 없습니다. ECE는 0.0050~0.0096로 기준선(0.0061) 대비 batch마다 등락하며, Batch 2에서 가장 크게 올랐지만(+0.0035) 같은 batch의 AUC는 기준선과 거의 동일합니다(+0.0002). 이는 위험 순위는 유지되지만 확률 보정만 소폭 흔들린 신호로, 변별력 저하(AUC 하락)와는 다른 성격입니다.

세 층위를 함께 보면 해석은 일관됩니다. PSI가 가장 크게 흔들린 Batch 2(`PD_calibrated` PSI 0.0011)에서 ECE도 함께 상승해, 입력 신호와 성능 신호가 같은 방향으로 움직였습니다. tier 비중 변화가 가장 컸던 Batch 3(월수입 PSI 0.0012, tier 변화 0.5702%p)에서는 AUC가 가장 낮았지만(-0.0064), 두 값 모두 각자의 위험 기준을 크게 밑돌아 정책과 성능이 동시에 흔들렸다고 보기는 어렵습니다. 반대로 PSI, tier, AUC 중 어느 하나만 단독으로 움직이는 패턴은 관찰되지 않았습니다. 따라서 정책 threshold 문제나 모델 드리프트를 개별 원인으로 의심할 근거도 약합니다.”

성능 저하 기준을 초과한 batch가 없고, PSI·tier·AUC/KS/ECE가 서로 다른 원인을 가리키지 않으므로, 즉시 재학습이나 정책 조정은 필요하지 않습니다. 다만 Batch 2 유형이 반복되면 재보정 주기를 앞당겨야 할 신호로, Batch 3 유형이 반복되면 재학습을 검토해야 할 신호로 해석할 수 있습니다. 따라서 두 패턴을 구분해 추적하는 모니터링 체계를 유지하는 것이 적절합니다.

## Step 5. 재학습 트리거 종합 판정

이 단계의 목적은 재학습 여부를 판단할 트리거 규칙을 남기는 데 있습니다. 단일 지표의 일시적 하락만으로 결정하지 않고, 입력 분포, 정책 분포, 성능 저하가 같은 방향으로 반복되는지를 함께 봅니다.

네 지표는 각각 다른 층위를 봅니다. PSI는 입력 분포 변화(신청자 구성이 학습 시점과 달라졌는가), tier 비중 변화는 정책 분포 변화(현재 threshold가 여전히 적절한가), AUC·KS는 모델의 변별력(위험 순위를 여전히 잘 맞히는가), ECE는 보정 성능(예측 확률의 신뢰도가 유지되는가)을 나타냅니다. 이 네 층 중 어디서 신호가 확인되는지에 따라 필요한 조치가 달라집니다. PSI만 변하면 원인 파악, tier만 변하면 정책 재검토, AUC·KS·ECE가 함께 흔들리면 재학습을 검토합니다.

5개 batch 전 구간에서 PSI·tier 변화·AUC·KS·ECE 모두 위험 기준을 하회하여, 즉각적 조치는 필요하지 않았습니다. 다만 Batch 2는 PSI와 ECE가 함께 상승해 재보정 신호에 가까웠고, Batch 3은 입력 분포 변화와 AUC 하락이 겹쳐 재학습 검토 신호에 가까웠습니다. 두 경우 모두 절대적인 폭이 기준을 크게 밑돌아, 이번 관찰 기간에는 즉시 조치가 필요한 수준으로 보기는 어렵습니다.

Batch 2 유형(PSI 상승 + ECE 상승 동반)이나 Batch 3 유형(입력 분포 변화 + AUC 하락 동반)이 한 번에 그치지 않고 반복적으로 관찰될 때만 각각 재보정 주기 단축, 재학습 검토로 연결합니다. 단발성으로 나타나는 경우에는 재학습이나 정책 조정 없이 모니터링을 유지하는 것을 원칙으로 합니다.

In [6]:
trigger_status = pd.DataFrame({
    'PSI_초과': psi_flag,
    'tier_쏠림_초과': tier_flag,
    '성능_저하_초과': perf_flag,
}).reindex(batch_sizes.index)
trigger_status['재학습_권고'] = trigger_status['성능_저하_초과']
trigger_status['정책_재검토_권고'] = trigger_status['tier_쏠림_초과'] & ~trigger_status['성능_저하_초과']
trigger_status['관찰_강화_권고'] = trigger_status['PSI_초과'] & ~trigger_status['tier_쏠림_초과'] & ~trigger_status['성능_저하_초과']

print(trigger_status.to_string())

overall_retrain_needed = trigger_status['재학습_권고'].any()
print(f"\n전체 관찰 기간 중 재학습 트리거 발동: {'예' if overall_retrain_needed else '아니오'}")


         PSI_초과  tier_쏠림_초과  성능_저하_초과  재학습_권고  정책_재검토_권고  관찰_강화_권고
batch                                                             
Batch 1   False       False     False   False      False     False
Batch 2   False       False     False   False      False     False
Batch 3   False       False     False   False      False     False
Batch 4   False       False     False   False      False     False
Batch 5   False       False     False   False      False     False

전체 관찰 기간 중 재학습 트리거 발동: 아니오


## Step 6. 산출물 저장

이 프로젝트의 마지막 노트북이므로, 파이프라인 전체가 실제 운영에서 어떻게 이어지는지 보여줄 수 있도록 batch별 3층 지표와 최종 트리거 판정을 저장합니다.


In [7]:
import json as _json

monitoring_summary = psi_summary.join(tier_drift_summary).join(perf_summary[['부도율', 'AUC', 'KS', 'ECE']])
monitoring_summary.to_csv('../data/monitoring_summary.csv')

retrain_trigger_status = {
    'baseline': {'AUC': BASELINE_AUC, 'KS': BASELINE_KS, 'ECE': BASELINE_ECE},
    'thresholds': {
        'PSI': 0.25, 'tier_shift_pct': TIER_SHIFT_THRESHOLD * 100,
        'AUC_drop': AUC_DROP_THRESHOLD, 'ECE_rise': ECE_RISE_THRESHOLD,
    },
    'overall_retrain_needed': bool(overall_retrain_needed),
    'batch_status': trigger_status.astype(bool).to_dict(orient='index'),
}
with open('../data/retrain_trigger_status.json', 'w', encoding='utf-8') as f:
    _json.dump(retrain_trigger_status, f, ensure_ascii=False, indent=2)

print(f"저장 완료: ../data/monitoring_summary.csv ({len(monitoring_summary)}행)")
print(f"저장 완료: ../data/retrain_trigger_status.json")


저장 완료: ../data/monitoring_summary.csv (5행)
저장 완료: ../data/retrain_trigger_status.json


## 결론

이 노트북에서 확정한 것은 다음과 같습니다.

- **Pseudo Production 구성 방식**: 절대 시점 정보가 없는 한계를 인정하고, `SK_ID_CURR` 순서를 시간 축 대안으로 사용해 `val_raw`를 5개 배치로 분할
- **3층 모니터링**: 입력 분포(PSI, train_raw 기준) → 정책 라벨 쏠림(tier 비중, val_raw 전체 기준) → 지연 라벨 성능(AUC/KS/ECE, 04 기준선 대비)을 서로 다른 층으로 분리해 추적
- **트리거 체계**: 층별로 의미가 다르므로 대응도 다르게 설계 — PSI만 초과하면 관찰 강화, tier 쏠림만 초과하면 정책 재검토, 성능 저하가 확인되면 재학습

이로써 `00_lgd_estimation.ipynb`부터 이어진 파이프라인 — LGD 시나리오 확정 → 데이터 마트 구성 → 정책 변수 검증 → PD 모델링 → Calibration → 정책 시뮬레이션 → 운영 모니터링 — 이 하나의 의사결정 흐름으로 연결합니다. 이 노트북들이 함께 보여주는 것은 단일 모델의 예측 성능이 아니라, 예측을 신뢰 가능한 확률로 만들고 그 확률을 실제 승인·한도 정책으로 전환하며 운영 이후의 성능과 정책 안정성까지 함께 관리하는 과정입니다.
